# HW4 評量推理模型：驗證器、MATH-500 與測試時擴展

> 說明：<https://github.com/chang-ye-tu/genai/blob/main/hw/hw4.md>　取材：Raschka《Build a Reasoning Model (From Scratch)》第 2–4 章（套件 `reasoning-from-scratch`）
> 核心段落：第 0–3 節與第 5 節（實作測驗只出這些段落的題目）；第 4 節為選做，不計分。
> 授權：本筆記本呼叫並改寫 Sebastian Raschka 的開源專案 <https://github.com/rasbt/reasoning-from-scratch>（Apache-2.0，© Sebastian Raschka）的範例程式；改寫部分（本課程的講解、問題與資料）同樣以 Apache-2.0 散布，完整授權文本見 repo 的 `LICENSES/Apache-2.0.txt`。
> 做法：**執行階段 → 變更執行階段類型 → T4 GPU**，由上而下逐格執行；看到「✍️ 請回答」就把觀察寫進該文字格。全部跑完後「檔案 → 下載 → .ipynb」上傳 iLearn，再作答實作測驗。
> **請勿更改模型名稱、版本、隨機種子與資料檔**，否則實作測驗的數值題會對不上。
> 需要 T4 GPU（0.6B 模型在 CPU 上太慢）。兩個 Qwen3-0.6B 權重各約 1.5 GB（共約 3 GB），下載後會驗證 SHA-256。


In [ ]:
import hashlib
def sha256_of(path, expected=None):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    digest = h.hexdigest()
    if expected is not None and digest != expected:
        raise RuntimeError(f"{path} 的 SHA-256 與課程固定版本不符：{digest[:12]}… ≠ {expected[:12]}…；請刪除檔案重新下載，或到 Q&A 回報")
    print(f"SHA-256 OK：{path} ({digest[:12]}…)")
    return digest

def fetch(path, urls, expected):
    # 依序嘗試 urls：每個網址下載後立刻驗 SHA-256，不符就刪掉換下一個來源；已存在且正確的檔案直接使用；全部來源都失敗才報錯
    import os, requests
    urls = [urls] if isinstance(urls, str) else list(urls)
    def _ok():
        h = hashlib.sha256()
        with open(path, "rb") as f:
            for chunk in iter(lambda: f.read(1 << 20), b""):
                h.update(chunk)
        return h.hexdigest() == expected
    if os.path.exists(path):
        if _ok():
            print(f"SHA-256 OK：{path} ({expected[:12]}…)"); return path
        print(f"{path} 的 SHA-256 與課程固定版本不符，刪除後重新下載"); os.remove(path)
    for url in urls:
        try:
            with requests.get(url, timeout=600, stream=True) as r:
                r.raise_for_status()
                with open(path, "wb") as f:
                    for chunk in r.iter_content(1 << 20):
                        f.write(chunk)
        except Exception as e:
            print("下載失敗：", url, e)
            if os.path.exists(path): os.remove(path)
            continue
        if _ok():
            print("下載自", url); print(f"SHA-256 OK：{path} ({expected[:12]}…)"); return path
        print(f"{url} 下載的內容 SHA-256 不符，改用下一個來源"); os.remove(path)
    raise RuntimeError(f"{path} 所有來源都無法取得正確的檔案（下載失敗或 SHA-256 不符）；請到 Q&A 回報")

# @title 第 0 節：安裝與載入 Qwen3-0.6B（base 版）
%pip -q install --no-deps reasoning-from-scratch==0.2.0
%pip -q install tokenizers==0.23.2 sympy==1.14.0
import torch, time, json, collections
from pathlib import Path
from reasoning_from_scratch.ch02 import get_device, generate_text_basic_cache, generate_stats
from reasoning_from_scratch.qwen3 import Qwen3Model, QWEN_CONFIG_06_B, Qwen3Tokenizer
device = get_device()
import platform, importlib.metadata as _meta
def _v(p):
    try: return _meta.version(p)
    except Exception: return "missing"  # metadata 查不到時印 missing；若同一格更早的 import 已失敗，程式到不了這裡，check_submissions 會判「無版本資訊／執行錯誤」
print("VERSIONS", "python=" + platform.python_version(), "torch=" + torch.__version__, *[p + "=" + _v(p) for p in ["reasoning-from-scratch", "tokenizers", "sympy"]])
# 權重與 tokenizer 都從固定的 Hugging Face commit 下載並驗證 SHA-256（不用套件的 download_qwen3_small，它抓的是 main，會漂移）
QWEN3_URL = "https://huggingface.co/rasbt/qwen3-from-scratch/resolve/8a3862bde98adf1c10e551bc4a400bf836f11e78/"
Path("qwen3").mkdir(exist_ok=True)
fetch("qwen3/qwen3-0.6B-base.pth", [QWEN3_URL + "qwen3-0.6B-base.pth", "https://f001.backblazeb2.com/file/reasoning-from-scratch/qwen3-0.6B/qwen3-0.6B-base.pth"], "ff38e5cc1530161ca5d68f6284f9343e58d637358b2bee9cc37e781b1c440443")
fetch("qwen3/tokenizer-base.json", [QWEN3_URL + "tokenizer-base.json", "https://f001.backblazeb2.com/file/reasoning-from-scratch/qwen3-0.6B/tokenizer-base.json"], "c0382117ea329cdf097041132f6d735924b697924d6f6fc3945713e96ce87539")
tokenizer = Qwen3Tokenizer(tokenizer_file_path="qwen3/tokenizer-base.json")
model = Qwen3Model(QWEN_CONFIG_06_B)
model.load_state_dict(torch.load("qwen3/qwen3-0.6B-base.pth", map_location="cpu"))
if device.type == "cuda" and not torch.cuda.is_bf16_supported():
    model.to(torch.float32)  # T4 不支援 bf16，改用 fp32（0.6B 約 2.4 GB）
model.to(device).eval()
n = sum(p.numel() for p in model.parameters())
print(f"參數量（本實作，輸出層與詞嵌入分開計）：{n:,} | 不含輸出層：{n - model.out_head.weight.numel():,}")
print("eos token：", tokenizer.eos_token, tokenizer.eos_token_id)


## 第 1 節 用自己實作的 Qwen3 生成文字

這個 Qwen3 是套件用純 PyTorch 從零寫的（GQA、RoPE、RMSNorm、KV cache），權重是官方的 Qwen3-0.6B **base** 模型：只做過預訓練，沒有對話或推理後訓練。


In [ ]:
prompt = "Explain large language models in a single sentence."
input_ids = torch.tensor(tokenizer.encode(prompt), device=device).unsqueeze(0)
print("提示的 token 數：", input_ids.shape[1])
t0 = time.time()
out = generate_text_basic_cache(model, input_ids, max_new_tokens=60, eos_token_id=tokenizer.eos_token_id)
generate_stats(out, tokenizer, t0, time.time())
print(tokenizer.decode(out.squeeze(0).tolist()))


✍️ **請回答 1-1**：base 模型的回答像「對話」還是「文章接龍」？第 5 節會載入同一個 base 系列、同一架構、同樣 0.6B 的推理版本（完整的後訓練系統：權重加上配套的對話模板與 4 個特殊 token），到時再回來比較：差異來自哪個訓練階段（第 9、10 單元）？（HW1 的 Qwen2.5-1.5B-Instruct 家族、規模與語料都不同，不能直接拿來當對照組。）

（在這裡作答）


## 第 2 節 驗證器：怎麼自動判斷數學答案對不對

評量推理模型不能只比字串：`0.5`、`1/2`、`\frac{1}{2}` 都是同一個數。套件的 `grade_answer` 會把兩邊正規化、用 sympy 比較。下面的表格請仔細看哪些會被判為相等——其中有一組（`10` vs `10%`）被判相等其實是**誤判**：正規化時把 `%` 去掉了，10 與 10% 並不是同一個數。驗證器也會犯錯，這就是第 6 單元說的「評量工具本身也要被評量」。


In [ ]:
from reasoning_from_scratch.ch03 import grade_answer, extract_final_candidate, render_prompt
tests = [("0.5", "1/2"), ("\\frac{1}{2}", "0.5"), ("3", "3.0"), ("x+1", "1+x"),
         ("(3, \\frac{\\pi}{2})", "\\left( 3, \\frac{\\pi}{2} \\right)"), ("10", "10%"),
         ("\\boxed{42} is the answer", "42"), ("12", "21"), ("2/3", "0.666"), ("\\sqrt{2}", "1.4142")]
for pred, gt in tests:
    print(f"{pred!r:<32} vs {gt!r:<34} → {grade_answer(pred, gt)}")
print()
print("extract_final_candidate('The answer is \\boxed{7}.') →", repr(extract_final_candidate("The answer is \\boxed{7}.")))
print("extract_final_candidate('So the result is 12 and then 15') →", repr(extract_final_candidate("So the result is 12 and then 15")))
print("\n提示模板：\n" + render_prompt("Compute 1/2 + 1/6."))


✍️ **請回答 2-1**：為什麼 `2/3` 與 `0.666` 被判為不相等，但 `0.5` 與 `1/2` 相等？`\boxed{42} is the answer` 為什麼要先經過 extract 才能判對？這對「用程式自動評分」的可靠性有什麼啟示？

（在這裡作答）


## 第 3 節 在 MATH-500 子集上評量 base 模型

MATH-500 有 500 題、5 個難度等級、7 個科目。這裡只跑前 `N_PROBLEMS` 題（每題最多 512 個 token，T4 約 4–6 分鐘）。


In [ ]:
from reasoning_from_scratch.ch03 import load_math500_test, evaluate_math500_stream
fetch("math500_test.json", "https://raw.githubusercontent.com/rasbt/reasoning-from-scratch/20e5e374544cb287f8acddcd9a6394507ac4f4e6/ch03/01_main-chapter-code/math500_test.json", "a9eccff7e25ecaf3952614e5b92fe081b4f0549c68ae6a387b96e131df9ddf41")
math_data = load_math500_test("math500_test.json")  # 檔案已存在且 hash 正確，套件只會讀取，不會再從 main 下載
print("題數：", len(math_data), "| 欄位：", list(math_data[0].keys()))
print("難度分佈：", dict(sorted(collections.Counter(d["level"] for d in math_data).items())))
print("科目分佈：", dict(collections.Counter(d["subject"] for d in math_data)))
print("第 1 題：", math_data[0]["problem"][:100], "… | 答案：", math_data[0]["answer"])
N_PROBLEMS = 10
MAX_NEW_TOKENS = 512
num_correct, num_examples, acc = evaluate_math500_stream(model, tokenizer, device, math_data[:N_PROBLEMS], out_path="math500-base.jsonl", max_new_tokens=MAX_NEW_TOKENS)


In [ ]:
# 看看模型「怎麼錯」：印出前 3 題的抽取答案與正確答案
for line in list(open("math500-base.jsonl", encoding="utf-8"))[:3]:
    r = json.loads(line)
    print(f"[{r['index']}] 正確={r['correct']} | 抽取={r['extracted']!r} | 標準={r['gtruth_answer']!r}")
    print("   生成（前 200 字）：", r["generated_text"][:200].replace("\n", " "), "\n")


✍️ **請回答 3-1**：base 模型答對幾題？錯的題目是「算錯」還是「格式不對（沒寫 \boxed{}）」？第 6 單元說評量常被格式干擾，這裡看到了嗎？

（在這裡作答）


## 第 4 節（選做）測試時擴展：多想幾次再投票（self-consistency）

同一題用 temperature 抽樣 `NUM_SAMPLES` 次，各自抽出最終答案後投票。多花推論算力，換取正確率——這是第 3、10 單元講的「測試時擴展」最簡單的形式。


In [ ]:
from reasoning_from_scratch.ch04 import self_consistency_vote
NUM_SAMPLES = 5
idx = 0
row = math_data[idx]
result = self_consistency_vote(model, tokenizer, render_prompt(row["problem"]), device,
                               num_samples=NUM_SAMPLES, temperature=0.8, top_p=0.9, max_new_tokens=MAX_NEW_TOKENS, show_progress=True, seed=123)
print("各次答案：", result["short_answers"])
print("投票結果：", result["final_answer"], "| 標準答案：", row["answer"], "| 投票是否正確：", grade_answer(result["final_answer"] or "", row["answer"]))


✍️ **請回答 4-1**：五次抽樣的答案一致嗎？投票結果比第 3 節的單次 greedy 好嗎？這個方法的成本是幾倍？在什麼情況下投票反而沒有幫助？

（在這裡作答）


## 第 5 節 對照：經過推理後訓練的 Qwen3-0.6B

同一個 base 系列（Qwen3-0.6B-Base → Qwen3-0.6B）、同一個架構、同樣 0.6B。比較的對象是**完整的後訓練系統**：除了權重，還包括配套的對話模板與 4 個特殊 token（`<think>` 等），所以這是「base 系統 vs 後訓練系統」的對照，不是只改權重的單變因實驗（tokenization、提示格式與 eos 都不同）；但比第 1 節與 HW1 的 Qwen2.5-Instruct 比較乾淨得多（那個混雜了家族、規模與語料）。推理版會先輸出 `<think>` 思考再作答。比較前 3 題的正確率與回答長度。


In [ ]:
fetch("qwen3/qwen3-0.6B-reasoning.pth", [QWEN3_URL + "qwen3-0.6B-reasoning.pth", "https://f001.backblazeb2.com/file/reasoning-from-scratch/qwen3-0.6B/qwen3-0.6B-reasoning.pth"], "35f6480cf58bb307cf193bfaf0c231d5178ffda08680565f8f0671d281382a43")
fetch("qwen3/tokenizer-reasoning.json", [QWEN3_URL + "tokenizer-reasoning.json"], "aeb13307a71acd8fe81861d94ad54ab689df773318809eed3cbe794b4492dae4")  # 備援站沒有這個檔（2026-09-05 為 404），只用固定 commit 的 Hugging Face
tok_r = Qwen3Tokenizer(tokenizer_file_path="qwen3/tokenizer-reasoning.json", apply_chat_template=True, add_generation_prompt=True, add_thinking=True)
model_r = Qwen3Model(QWEN_CONFIG_06_B)
model_r.load_state_dict(torch.load("qwen3/qwen3-0.6B-reasoning.pth", map_location="cpu"))
if device.type == "cuda" and not torch.cuda.is_bf16_supported():
    model_r.to(torch.float32)
model_r.to(device).eval()
N_REASONING = 3
_ = evaluate_math500_stream(model_r, tok_r, device, math_data[:N_REASONING], out_path="math500-reasoning.jsonl", max_new_tokens=1024)
r = json.loads(open("math500-reasoning.jsonl", encoding="utf-8").readline())
print("\n推理模型第 1 題的生成（前 400 字）：", r["generated_text"][:400].replace("\n", " "))


✍️ **請回答 5-1**：推理模型的回答長度和 base 模型差多少倍？正確率有沒有提高？「先想再答」和第 4 節「多答幾次再投票」都是花更多 token 換正確率，兩者有什麼本質差別？

（在這裡作答）


## ✍️ AI 使用聲明（必填）

| 項目 | 內容 |
|------|------|
| 使用的工具 | （例如：ChatGPT 免費版、Colab 內建 Gemini） |
| 用在哪些工作 | （例如：解釋錯誤訊息、幫我看懂某一格程式） |
| 我自己完成的部分 | （例如：全部執行、所有 ✍️ 回答） |
| 我如何驗證 AI 的說法 | （例如：實際執行、對照投影片） |

姓名／學號：
